# 01 — Data Engineering

**P.U.L.S.E. — Predictive Unified Life-sciences Summarization Engine**

This notebook walks through the complete data pipeline:

1. Raw data inventory across 5 source datasets
2. Per-source loading, cleaning and feature extraction
3. Master patient table construction (127 k rows × 33 features)
4. Feature quality report & missing-value analysis
5. Export to Parquet

In [ ]:
import os, sys
ROOT = os.path.dirname(os.path.abspath('.'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
print('Environment ready')

## 1  Raw Data Inventory

In [ ]:
from src.data.loader import (
    load_nhanes_demographics, load_nhanes_labs, load_nhanes_diabetes,
    load_nhanes_hba1c, load_chronic_disease, load_diabetes_uci,
    load_drug_reviews, load_statlog_heart
)

sources = {
    'NHANES demographics 2015': load_nhanes_demographics('2015-16'),
    'NHANES labs 2015':         load_nhanes_labs('2015-16'),
    'NHANES demographics 2021': load_nhanes_demographics('2021-23'),
    'NHANES labs 2021':         load_nhanes_labs('2021-23'),
    'Chronic Disease':          load_chronic_disease(),
    'Diabetes UCI':             load_diabetes_uci(),
    'Drug Reviews':             load_drug_reviews(),
    'Statlog Heart':            load_statlog_heart(),
}

inv = pd.DataFrame([
    {'Source': k, 'Rows': len(v), 'Columns': v.shape[1]}
    for k, v in sources.items()
])
print(inv.to_string(index=False))

## 2  Feature Store — build master table

In [ ]:
from src.data.feature_store import build_feature_store

master = build_feature_store()
print(f'Master table: {master.shape[0]:,} rows × {master.shape[1]} columns')
master.head()

## 3  Missing-value analysis

In [ ]:
missing = (
    master.isnull().sum()
          .to_frame('missing_count')
          .assign(pct=lambda d: d['missing_count'] / len(master) * 100)
          .sort_values('pct', ascending=False)
)
missing = missing[missing['missing_count'] > 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(missing.index, missing['pct'], color=sns.color_palette('muted')[0])
ax.set_xlabel('Missing (%)')
ax.set_title('Missing values per feature')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.show()

print(missing.to_string())

## 4  Feature distributions — clinical features

In [ ]:
clinical_cols = ['age', 'bmi', 'glucose', 'hba1c',
                 'blood_pressure_sys', 'blood_pressure_dia', 'serum_creatinine']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(clinical_cols):
    ax = axes[i]
    data = master[col].dropna()
    ax.hist(data, bins=40, edgecolor='white', color=sns.color_palette('muted')[i % 6])
    ax.set_title(col)
    ax.set_xlabel('')
    ax.set_ylabel('count')

axes[-1].axis('off')
fig.suptitle('Clinical Feature Distributions — Master Table', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5  Source breakdown

In [ ]:
src_counts = master['source'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
src_counts.plot.barh(ax=ax, color=sns.color_palette('muted')[:len(src_counts)])
ax.set_xlabel('Rows')
ax.set_title('Rows per data source')
for p in ax.patches:
    ax.text(p.get_width() + 200, p.get_y() + p.get_height() / 2,
            f'{p.get_width():,.0f}', va='center')
plt.tight_layout()
plt.show()